In [ ]:
# instalar librerias usando uv pip install -r requirements.txt

In [1]:
import os
import fiftyone as fo
import fiftyone.zoo as foz
import optuna
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import random
from dotenv import load_dotenv


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%matplotlib inline

In [ ]:
%load_ext tensorboard

In [ ]:

%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir /home/mikel/github/TKNIKA/kortxovision/yolo/runs


In [ ]:
load_dotenv(override=True)

In [ ]:
fo.config.dataset_zoo_dir = os.getenv("DATA_FOLDER")
train_dataset_name = os.getenv("TRAIN_DATASET_NAME")
val_dataset_name = os.getenv("VAL_DATASET_NAME")
test_dataset_name = os.getenv("TEST_DATASET_NAME")
yolo_train_folder = os.getenv("YOLO_TRAIN_FOLDER")
yolo_best_model_path = os.getenv("YOLO_BEST_MODEL_PATH")
datasets = [train_dataset_name, val_dataset_name, test_dataset_name]
print(datasets)

In [ ]:
if datasets in fo.list_datasets():
    fo.delete_dataset(datasets)
    print(f"Dataset '{datasets}' antiguo encontrado y eliminado.")

In [ ]:
for name in datasets:
    if name in fo.list_datasets():
        fo.delete_dataset(name)
        print(f" - Dataset '{name}' antiguo eliminado.")
print("¡Limpieza completada!\n")

In [ ]:
num_samples_train = 1500
num_samples_val = 150
num_samples_test = 150
train_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    dataset_name=train_dataset_name,
    split="train",
    label_types=["detections"],
    classes=["Cat", "Dog"],
    max_samples=num_samples_train,
    persistent=False,
    #download_if_necessary=False
)

val_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    dataset_name=val_dataset_name,
    split="validation",
    label_types=["detections"],
    classes=["Cat", "Dog"],
    max_samples=num_samples_val,
    persistent=False,
    #download_if_necessary=False
)

test_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    dataset_name=test_dataset_name,
    split="test",
    label_types=["detections"],
    classes=["Cat", "Dog"],
    max_samples=num_samples_test,
    persistent=False,
    #download_if_necessary=False
)


In [ ]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


In [ ]:
from fiftyone import ViewField 
cat_view_train  = train_dataset.match(ViewField("ground_truth.detections.label").contains("Cat"))
dog_view_train = train_dataset.match(ViewField("ground_truth.detections.label").contains("Dog"))

print(len(cat_view_train))
print(len(dog_view_train))


cat_view_val  = val_dataset.match(ViewField("ground_truth.detections.label").contains("Cat"))
dog_view_val = val_dataset.match(ViewField("ground_truth.detections.label").contains("Dog"))

print(len(cat_view_val))
print(len(dog_view_val))

cat_view_test  = test_dataset.match(ViewField("ground_truth.detections.label").contains("Cat"))
dog_view_test = test_dataset.match(ViewField("ground_truth.detections.label").contains("Dog"))

print(len(cat_view_test))
print(len(dog_view_test))





In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
train_first_sample = train_dataset.first()  
val_first_sample = val_dataset.first()
test_first_sample = test_dataset.first()
# Train image
train_img = Image.open(train_first_sample.filepath)
axes[0].imshow(train_img)
axes[0].set_title(f"Train Sample\n{os.path.basename(train_first_sample.filepath)}")
axes[0].axis('off')

# Validation image
val_img = Image.open(val_first_sample.filepath)
axes[1].imshow(val_img)
axes[1].set_title(f"Validation Sample\n{os.path.basename(val_first_sample.filepath)}")
axes[1].axis('off')

# Test image
test_img = Image.open(test_first_sample.filepath)
axes[2].imshow(test_img)
axes[2].set_title(f"Test Sample\n{os.path.basename(test_first_sample.filepath)}")
axes[2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
classes = ["Cat", "Dog"]
train_dataset.export(
    export_dir="data/yolo/train",
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth", 
    split="train",          
    classes=classes,
)


val_dataset.export(
    export_dir="data/yolo/val",
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth", 
    split="validation",        
    classes=classes,
)

test_dataset.export(
    export_dir="data/yolo/test",
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",  # or your label field
    split="test",      
    classes=classes,   
)



In [ ]:
classes = ["Cat", "Dog"]
print(classes)

In [ ]:
model = YOLO('yolov8n.pt')


results = model.train(
    data=yolo_train_folder, 
    epochs=100,
    imgsz=640,
    project='yolo/runs', 
    lr0=0.001,
    lrf=0.2,
    name='100epochs'
)

In [ ]:
%load_ext tensorboard

In [ ]:

model = YOLO(yolo_best_model_path)


metrics = model.val()


print("Resultados en el conjunto de Test:")
print(f"mAP50-95: {metrics.box.map}")
print(f"mAP50: {metrics.box.map50}")
print(f"Precisión: {metrics.box.p[0]}") 
print(f"Recall: {metrics.box.r[0]}")    

In [ ]:
test_images_dir = 'data/yolo/test/images/test'
model = YOLO(yolo_best_model_path)
try:
    model = YOLO(yolo_best_model_path)
    print("Modelo cargado correctamente desde:", yolo_best_model_path)
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    print("Por favor, asegúrate de que la ruta al archivo 'best.pt' es correcta.")

try:
    image_files = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    if not image_files:
        raise FileNotFoundError("No se encontraron archivos de imagen en el directorio de test.")

    random_image_name = random.choice(image_files)
    random_image_path = os.path.join(test_images_dir, random_image_name)
    
    print(f"Imagen seleccionada para inferencia: {random_image_path}")

    results = model.predict(random_image_path)

    annotated_image = results[0].plot()
    annotated_image_rgb = annotated_image[:, :, ::-1]

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_image_rgb)
    plt.title(f'Predicciones en: {random_image_name}')
    plt.axis('off')
    plt.show()

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Por favor, revisa la ruta a tu carpeta de imágenes de test.")
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")

to train a a script use
```bash
yolo train model=yolov8s.pt data='dataset.yaml' epochs=110 imgsz=640 project='yolo/runs' lr0=0.001 lrf=0.2 name='110epochs'
```